# **MobileNet w/o XAI**

In [1]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
NVIDIA GeForce RTX 4090


In [ ]:
"""
Baseline MobileNetV2 for 5-class knee-X-ray classification.

Directory layout:
E:/FA011/MSC/SplittedDataset/{train|val|test}/<class_name>/*.png|*.jpg
"""

# ───────────────────── 1. IMPORTS ─────────────────────
import os, cv2, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.mobilenet_v2 import (
    MobileNetV2, preprocess_input
)
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

from sklearn.metrics import (
    confusion_matrix, classification_report, precision_score,
    recall_score, f1_score, roc_curve, auc
)
from sklearn.preprocessing import label_binarize

# ───────────────────── 2. PARAMETERS ──────────────────
TRAIN_DIR = '/mnt/d/FA019/new/SplittedDataset/train'
VAL_DIR   ='/mnt/d/FA019/new/SplittedDataset/val'
TEST_DIR  = '/mnt/d/FA019/new/SplittedDataset/test'

IMG_SIZE   = (224, 224)          # MobileNetV2 native size
BATCH_SIZE = 32
NUM_CLASSES = len(os.listdir(TRAIN_DIR))

# ────────────── 3. BPHE + PREPROCESS ────────────────
def bphe_rgb(img):
    img = img.astype(np.uint8) if img.max() > 1 else (img * 255).astype(np.uint8)
    y, cr, cb = cv2.split(cv2.cvtColor(img, cv2.COLOR_RGB2YCrCb))
    y_eq = cv2.equalizeHist(y)
    img_eq = cv2.cvtColor(cv2.merge((y_eq, cr, cb)), cv2.COLOR_YCrCb2RGB)
    return preprocess_input(img_eq.astype(np.float32))

# ───────────── 4. DATA GENERATORS ───────────────
train_gen = ImageDataGenerator(
    preprocessing_function=bphe_rgb,
    rotation_range=30, width_shift_range=0.2, height_shift_range=0.2,
    shear_range=0.2, zoom_range=0.2, horizontal_flip=True,
    brightness_range=[0.8, 1.2], fill_mode='nearest'
).flow_from_directory(TRAIN_DIR, target_size=IMG_SIZE,
                      batch_size=BATCH_SIZE, class_mode='categorical')

val_gen = ImageDataGenerator(preprocessing_function=bphe_rgb
).flow_from_directory(VAL_DIR, target_size=IMG_SIZE,
                      batch_size=BATCH_SIZE, class_mode='categorical')

test_gen = ImageDataGenerator(preprocessing_function=bphe_rgb
).flow_from_directory(TEST_DIR, target_size=IMG_SIZE,
                      batch_size=BATCH_SIZE, class_mode='categorical',
                      shuffle=False)

# ───────────── 5. BUILD MODEL ───────────────
base = MobileNetV2(weights='imagenet', include_top=False,
                   input_shape=IMG_SIZE + (3,))

x = GlobalAveragePooling2D()(base.output)
out = Dense(NUM_CLASSES, activation='softmax', name='pred')(x)
model = Model(inputs=base.input, outputs=out)

# Freeze backbone
for layer in base.layers:
    layer.trainable = False

model.compile(optimizer=Adam(learning_rate=0.0001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

# ───────────── 6. TRAINING ───────────────
history = model.fit(
    train_gen,
    epochs=100,
    validation_data=val_gen
)

# ─────── 7. ACCURACY & LOSS PLOTS ────────
plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.plot(history.history['accuracy'], label='Train')
plt.plot(history.history['val_accuracy'], label='Val')
plt.title('Accuracy'); plt.legend()

plt.subplot(1,2,2)
plt.plot(history.history['loss'], label='Train')
plt.plot(history.history['val_loss'], label='Val')
plt.title('Loss'); plt.legend()
plt.tight_layout(); plt.show()

# ───────── 8. EVALUATION ─────────
test_loss, test_acc = model.evaluate(test_gen)
print(f"\nTest Accuracy : {test_acc:.4f}")
print(f"Test Loss     : {test_loss:.4f}")

test_gen.reset()
prob   = model.predict(test_gen)
y_pred = np.argmax(prob, axis=1)
y_true = test_gen.classes
class_names = list(test_gen.class_indices.keys())

conf_mat = confusion_matrix(y_true, y_pred)
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=class_names))
print(f"Precision (weighted): {precision_score(y_true, y_pred, average='weighted'):.4f}")
print(f"Recall    (weighted): {recall_score(y_true, y_pred, average='weighted'):.4f}")
print(f"F1 Score  (weighted): {f1_score(y_true, y_pred, average='weighted'):.4f}")

plt.figure(figsize=(10,7))
sns.heatmap(conf_mat, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names,
            annot_kws={'size':18})
plt.xlabel('Predicted'); plt.ylabel('True')
plt.title('MobileNetV2 Confusion Matrix')
plt.tight_layout(); plt.show()

# ───────── 9. ROC CURVE ─────────
y_true_bin = label_binarize(y_true, classes=list(range(NUM_CLASSES)))
fpr, tpr, roc_auc = {}, {}, {}
plt.figure(figsize=(10,8))
colors = ['blue','green','orange','red','purple']
for i, c in enumerate(colors):
    fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], prob[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])
    plt.plot(fpr[i], tpr[i], color=c, lw=2,
             label=f'Class {i} (AUC={roc_auc[i]:.2f})')

plt.plot([0,1],[0,1],'k--',lw=2)
plt.xlim([0,1]); plt.ylim([0,1.05])
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('Multi‑Class ROC Curve (MobileNetV2)')
plt.legend(loc='lower right'); plt.grid(True); plt.tight_layout(); plt.show()


2025-09-16 15:27:22.591861: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-09-16 15:27:22.600409: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1758014842.609162 1165442 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1758014842.612200 1165442 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1758014842.620709 1165442 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

Found 1614 images belonging to 5 classes.
Found 229 images belonging to 5 classes.
Found 466 images belonging to 5 classes.
